In [ ]:
from steer_core.DataManager import DataManager

from steer_materials.CellMaterials.Base import CurrentCollectorMaterial, InsulationMaterial, SeparatorMaterial
from steer_materials.CellMaterials.Electrode import CathodeMaterial, Binder, ConductiveAdditive, AnodeMaterial

from steer_opencell_design.Components.CurrentCollectors.Tabless import TablessCurrentCollector
from steer_opencell_design.Components.Electrodes import Cathode, Anode
from steer_opencell_design.Components.Separators import Separator
from steer_opencell_design.Constructions.Layups.Laminate import Laminate
from steer_opencell_design.Formulations.ElectrodeFormulations import CathodeFormulation, AnodeFormulation
from steer_opencell_design.Constructions.ElectrodeAssemblies.JellyRolls import WoundJellyRoll
from steer_opencell_design.Constructions.ElectrodeAssemblies.WindingEquipment import RoundMandrel

from steer_opencell_design import __version__

import pandas as pd
from datetime import datetime as dt

In [ ]:
db = DataManager()

In [ ]:
db.create_table(
    table_name='cells',
    columns={
        'name': 'TEXT PRIMARY KEY',
        'object': 'BLOB',
        'form_factor': 'TEXT',
        'internal_construction': 'TEXT',
        'date': 'TEXT',
        'version': 'TEXT',
        'reference': 'TEXT'
    }
)

In [ ]:
db.remove_data(table_name='cells', condition="name = 'CBAK-32140NS'")

In [ ]:
db.get_table_names()

In [ ]:
# set some standard materials

conductive_additive = ConductiveAdditive.from_database("Super P")
binder = Binder.from_database("PVDF")
insulation = InsulationMaterial.from_database("Aluminium Oxide, 95%")
current_collector_material = CurrentCollectorMaterial.from_database('Aluminum')
separator_material = SeparatorMaterial.from_database('Polyethylene')



In [ ]:
# Create the cathode

cathode_current_collector = TablessCurrentCollector(
    material=current_collector_material,
    width=130,
    length=3200,
    coated_width=125,
    insulation_width=2.5,
    thickness=13.5
)

cathode_active_material = CathodeMaterial.from_database("NFM111 (Vendor C)")

cathode_formulation = CathodeFormulation(
    active_materials={cathode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_cathode = Cathode(
    formulation=cathode_formulation,
    current_collector=cathode_current_collector,
    calender_density=2.93,
    mass_loading=11.27,
    insulation_material=insulation,
    insulation_thickness=3
)

my_cathode.properties

In [ ]:
# Create the anode

anode_current_collector = TablessCurrentCollector(
    material=current_collector_material,
    width=133,
    length=3250,
    coated_width=128,
    insulation_width=2.5,
    thickness=13.5,
)

anode_active_material = AnodeMaterial.from_database("Hard Carbon (Vendor A)")

anode_formulation = AnodeFormulation(
    active_materials={anode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_anode = Anode(
    formulation=anode_formulation,
    current_collector=anode_current_collector,
    calender_density=1.1,
    mass_loading=8,
    insulation_material=insulation,
    insulation_thickness=3
)

my_anode.properties

In [ ]:
# create the layup 

top_separator = Separator(
    material=separator_material, 
    thickness=12,
    width = 130,
    length = 3600
)

bottom_serparator = Separator(
    material=separator_material, 
    thickness=12,
    width = 130,
    length = 3600,
)

my_layup = Laminate(
    anode=my_anode,
    cathode=my_cathode,
    top_separator=top_separator,
    bottom_separator=bottom_serparator,
    name="CBAK-32140NS"
)



In [ ]:
# create the jellyroll assembly

mandrel = RoundMandrel(
    diameter=5,
    length=500,
)

my_jellyroll = WoundJellyRoll(
    laminate=my_layup,
    mandrel=mandrel,
    name="CBAK-32140NS"
)


In [ ]:
my_jellyroll.radius

In [ ]:
my_jellyroll.roll_properties

In [ ]:
my_jellyroll.get_spiral_plot(height=1300, width=1300).show()

In [ ]:

db.insert_data(table_name='cells', data=pd.DataFrame({
    'name': [my_jellyroll.name],
    'object': [my_jellyroll.serialize()],
    'form_factor': ['Cylindrical'],
    'internal_construction': ['Wound'],
    'date': [dt.now().strftime("%Y-%m-%d %H:%M:%S")],
    'version': [__version__],
    'reference': ['Na/Na+']
}))


In [ ]:
db.get_data('cells')